# Conditional Programming

When working with data, we often want our scripts to take different actions depending on the value of a variable or the result of a calculation. For example, we might label a student as passing or failing, flag an observation that falls outside an expected range, or handle a missing value differently from an observed value.

Conditional programming uses logical conditions, expressions that evaluate to `TRUE` or `FALSE` to control which code R runs. In this section, you will use `if`, `else`, and `else if` to make decisions for individual values. You will also learn how to handle missing values (`NA`) and use `ifelse()` to apply a two-way condition across an entire vector.

## If statements

An `if` statement is a control-flow construct: it allows a program to choose whether to run a block of code based on a condition.

Here is a simple example:

``` r
grade <- 51

if (grade >= 50) {
  print("Pass")
}
```

```text
[1] "Pass"
```

The condition inside the parentheses, `grade >= 50`, compares the value of `grade` with 50. As you know, comparison operators such as `>=` produce a logical value: either `TRUE` or `FALSE`.

- If the condition is `TRUE`, R runs the code inside the curly brackets `{}`.
- If the condition is `FALSE`, R skips that block of code.

In this example, `51 >= 50` is `TRUE`, so R prints `"Pass"`.

The version above only specifies what should happen when the condition is true. If `grade` were below 50, nothing would be printed:

To specify an alternative action when the condition is false, add an `else` block:

```r
grade <- 48

if (grade >= 50) {
  print("Pass")
} else {
  print("Fail")
}
```

```text
[1] "Fail"
```

Here, `48 >= 50` is `FALSE`, so R skips the first block and runs the code following `else`.

````{important}
Keep `else` on the same line as the closing curly bracket of the preceding `if` block:


```r
} else {
```

Putting `else` on a new line can cause R to interpret the `if` statement as finished, resulting in an `unexpected 'else'` error.
````

## else if

Grades do not always fall into only two categories. For example, we may want to classify a score as `"Fail"`, `"Pass"`, `"Merit"`, or `"Distinction"`.

An `if` / `else if` / `else` chain lets us choose one outcome from several possible alternatives:

```r
score <- 72

if (score >= 70) {
  grade <- "Distinction"
} else if (score >= 60) {
  grade <- "Merit"
} else if (score >= 50) {
  grade <- "Pass"
} else {
  grade <- "Fail"
}

grade
```

```text
[1] "Distinction"
```

R tests the conditions in order, from top to bottom:

1. Is `score >= 70`?
2. If not, is `score >= 60`?
3. If not, is `score >= 50`?
4. If none of these conditions are `TRUE`, use the final `else` block.

As soon as R finds a condition that is `TRUE`, it runs that block and exits the entire chain. The remaining conditions are not tested. Since `72 >= 70` is `TRUE`, R assigns `"Distinction"` to `grade` and does not consider the other categories.

The order of conditions matters. Test the highest threshold first. If the `"Pass"` condition, `score >= 50`, appeared first, a score of 72 would be classified as a pass before R had an opportunity to test whether it qualified for merit or distinction.





## Dealing with missing values

A missing value in R is represented by `NA`. It means that a value is unknown or unavailable; it is not the same as zero, an empty character string, or `FALSE`.

A logical comparison involving `NA` usually produces `NA`, rather than `TRUE` or `FALSE`:

```r
grade <- NA

grade >= 50
```

```text
[1] NA
```

This creates a problem for an `if` statement. The condition in an `if` statement must evaluate to exactly one `TRUE` or `FALSE` value. R cannot decide what to do when the condition is `NA`:

``` r
grade <- NA

if (grade >= 50) {
  print("Pass")
} else {
  print("Fail")
}
```

```text
Error in if (grade >= 50) { : missing value where TRUE/FALSE needed
```

To handle missing values explicitly, use `is.na()`. This function returns `TRUE` when its input is missing and `FALSE` otherwise.

```r
is.na(NA)

is.na(50)
```

```text
[1] TRUE
[1] FALSE
```

Place the `is.na()` check first in an `if` / `else if` / `else` chain:

```r
grade <- NA

if (is.na(grade)) {
  print("Missing")
} else if (grade >= 50) {
  print("Pass")
} else {
  print("Fail")
}
```

```text
[1] "Missing"
```

When `grade` is `NA`, `is.na(grade)` evaluates to `TRUE`. R prints `"Missing"` and exits the conditional chain, so it never attempts to evaluate `grade >= 50`.

Checking for missing values first is important whenever later conditions involve comparisons or calculations that would produce `NA`.



## Vectorised if

The `if` statements we have looked at so far are not vectorised: they expect a single `TRUE` or `FALSE` value. If you pass a vector of grades, the condition may produce several values, and `if` cannot decide which one to use.

```r
scores <- c(40, 50)

if (is.na(scores)) {
  print("Missing")
} else if (scores >= 50) {
  print("Pass")
} else {
  print("Fail")
}
```
This produces an error:

```text
Error in if (is.na(scores)) { : the condition has length > 1
```

Here, `is.na(scores)` returns `c(FALSE, FALSE)`, not a single logical value. The same issue occurs with `scores >= 50`, which also returns a logical vector.

For element-by-element decisions, use the vectorised `ifelse()` function. It takes a logical condition as its first argument, followed by the value to return where the condition is `TRUE` and the value to return where it is `FALSE`.

```r
scores <- c(30, 50, 70)

ifelse(scores >= 50, "Pass", "Fail")
```
```text
[1] "Fail" "Pass" "Pass"
```
A single `ifelse()` handles two outcomes. To handle more categories, you can nest calls: put another `ifelse()` in the `FALSE` argument. That call handles the elements that did not meet the previous condition.

```r
scores <- c(35, 58, 62, 72, NA)

ifelse(
  is.na(scores), "Missing",
  ifelse(
    scores >= 70, "Distinction",
    ifelse(
      scores >= 60, "Merit",
      ifelse(scores >= 50, "Pass", "Fail")
    )
  )
)
```
```text
[1] "Fail"        "Pass"        "Merit"       "Distinction" "Missing"
```
The order of the conditions matters: each score is assigned the first matching category. Checking `is.na(scores)` first ensures that missing grades are labelled "Missing" rather than passed through the later comparisons.